# Séance 11 · Exercices — Le RAG, donner de la mémoire à son IA · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

**Niveau de la séance : ⭐⭐⭐** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab**, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
Tous les exercices marchent en **mode démo** (`USE_MODEL = False`) et avec **TF-IDF** (pas besoin de sentence-transformers) : la recherche, le prompt et l'évaluation sont bien réels, seule la réponse finale est écrite à la main par le faux modèle.


## Préparation

La même cellule qu'à la leçon (elle prépare `llm(messages)`), plus les règles du jeu *Sardine Express*, les outils TF-IDF de scikit-learn, une liste de mots vides français (« le », « la », « qui »... qui ne disent rien du sujet) et le helper `verifier`. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import numpy as np
import matplotlib.pyplot as plt

# ---------- Mode démo : un faux LLM qui répond sans réseau ni GPU ----------
def llm_factice(messages):
    """Mode démo : sans contexte, le faux modèle invente ; avec contexte, il recopie le passage le plus utile."""
    q = messages[-1]["content"]
    ql = q.lower()
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    if "Contexte :" in systeme:                                   # un RAG lui a donné des passages
        contexte = systeme.split("Contexte :", 1)[1]
        mots_question = {m for m in re.findall(r"\w{4,}", ql)} - {"quel", "quelle", "quels", "comment", "combien", "pourquoi", "dans", "avec", "pour"}
        phrases = [p.strip(" -\n") for p in re.split(r"(?<=[.!?])\s+", contexte) if p.strip(" -\n")]
        meilleure = max(phrases, key=lambda p: sum(m in p.lower() for m in mots_question), default="")
        if meilleure and sum(m in meilleure.lower() for m in mots_question) > 0:
            return "D'après tes documents : " + meilleure
        return "Je ne trouve pas cette information dans les documents fournis."
    if "sardine" in ql:
        return "Sardine Express est un jeu de cartes rapide où chaque joueur doit se débarrasser de ses sardines avant les autres. Il se joue avec 52 cartes classiques."
    if "prof" in ql or "cours" in ql:
        return "Le cours a lieu le lundi matin, dans la salle 12, avec le professeur Dupont."
    return "Bonne question ! Je ne suis pas certain, mais voici une réponse plausible : c'est un sujet intéressant."

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

# ---------- Outils des exercices ----------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

REGLES_DU_JEU = """Sardine Express est un jeu de cartes pour 2 à 5 joueurs, à partir de 8 ans. Une partie dure environ 15 minutes.

Le jeu contient 60 cartes Sardine (numérotées de 1 à 12, en 5 couleurs), 8 cartes Mouette et 4 cartes Tempête. On utilise aussi un dé à six faces.

Au début de la partie, on mélange toutes les cartes. Chaque joueur reçoit 7 cartes, et le reste forme la pioche, face cachée. La carte du dessus de la pioche est retournée : c'est le début de la défausse.

Le joueur le plus jeune commence. Ensuite, on joue dans le sens des aiguilles d'une montre.

À son tour, un joueur doit poser une carte Sardine de la même couleur OU du même numéro que la carte du dessus de la défausse. S'il ne peut pas, il pioche une carte et son tour est terminé.

La carte Mouette peut être posée sur n'importe quelle carte. Le joueur suivant doit alors piocher 2 cartes et passer son tour. Deux Mouettes ne peuvent pas être posées l'une sur l'autre.

La carte Tempête inverse le sens du jeu. Si elle est posée, on lance le dé : sur un 6, tous les joueurs passent leur main entière à leur voisin de gauche.

Quand un joueur n'a plus qu'une seule carte en main, il doit crier « Sardine ! ». S'il oublie et qu'un autre joueur le remarque avant le tour suivant, il pioche 3 cartes de pénalité.

Le premier joueur qui n'a plus de cartes gagne la manche. Il marque 1 point par carte restante dans la main de chaque adversaire, et 5 points par Mouette restante.

Une partie complète se joue en 3 manches. Le joueur avec le plus de points à la fin des 3 manches gagne la partie. En cas d'égalité, on joue une manche de plus.

Variante « Banc de sardines » : si un joueur a en main 3 cartes du même numéro, il peut les poser d'un coup à son tour, quelle que soit la couleur.

Variante « Mode rapide » : chaque joueur reçoit 5 cartes au lieu de 7, et la partie se joue en une seule manche.

Si la pioche est vide, on mélange la défausse (sauf la carte du dessus) pour former une nouvelle pioche.

Il est interdit de regarder les cartes des autres joueurs. Un joueur surpris à tricher perd immédiatement la manche en cours.

Le jeu a été créé en 2019 par Léa Marchand et illustré par Tom Ravel. Il est édité par les éditions du Phare, à Brest."""

# Mots vides : ils sont dans toutes les phrases, on les ignore pour chercher par mots-clés
STOP_FR = "le la les l un une des du de d et ou à a au aux en dans sur par pour avec ce cet cette ces se son sa ses mon ma mes ton ta tes il elle on ne pas plus que qui quoi quel quelle quels est sont c'est".split()

REFUS = "Je ne trouve pas cette information dans les documents."

def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 2 phrases maximum."):
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}])

# ---------- Vérification automatique des exercices ----------
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Prêt :", len(REGLES_DU_JEU.split()), "mots de règles à indexer")

## Exercice 1 ⭐ · Découper en paragraphes

Première étape du RAG : **découper**. Écris `decouper(texte)` qui renvoie la liste des paragraphes (séparés par une ligne vide), sans espaces autour et sans paragraphe vide, puis applique-la aux règles du jeu.

Résultat attendu : 15 chunks, le premier commence par « Sardine Express », le dernier parle de Léa Marchand.

<details><summary>Indice</summary>

`texte.split("\n\n")` puis `.strip()` sur chaque morceau, en gardant seulement ceux qui ne sont pas vides.

</details>

In [ ]:
# À toi
def decouper(texte):
    return None

chunks = decouper(REGLES_DU_JEU)
print(len(chunks) if chunks else None, "chunks")

In [ ]:
verifier("Exercice 1 · 15 chunks", lambda: len(chunks) == 15 and all(c == c.strip() and c for c in chunks))
verifier("Exercice 1 · premier et dernier", lambda: chunks[0].startswith("Sardine Express") and "Léa Marchand" in chunks[-1])

<details><summary>Solution</summary>

```python
def decouper(texte):
    return [p.strip() for p in texte.split("\n\n") if p.strip()]

chunks = decouper(REGLES_DU_JEU)
print(len(chunks), "chunks")   # 15
for i, c in enumerate(chunks[:3]):
    print(f"[{i}] {c[:70]}...")
```

</details>

## Exercice 2 ⭐ · La similarité cosinus à la main

Écris `similarite(a, b)` **sans numpy** : le produit scalaire (somme des produits terme à terme) divisé par le produit des longueurs (racine carrée de la somme des carrés).

Résultat attendu : `similarite([1, 0], [1, 0])` = 1, `similarite([1, 0], [0, 1])` = 0, `similarite([1, 2, 3], [2, 4, 6])` = 1 (même direction, longueurs différentes), `similarite([1, 1], [1, 0])` ≈ 0,707.

<details><summary>Indice</summary>

`produit = sum(x * y for x, y in zip(a, b))`, `longueur_a = sum(x * x for x in a) ** 0.5`, idem pour b, puis `produit / (longueur_a * longueur_b)`.

</details>

In [ ]:
# À toi
def similarite(a, b):
    return None

print(similarite([1, 0], [1, 0]), similarite([1, 0], [0, 1]), similarite([1, 2, 3], [2, 4, 6]), similarite([1, 1], [1, 0]))

In [ ]:
verifier("Exercice 2 · même direction = 1", lambda: abs(similarite([1, 0], [1, 0]) - 1) < 1e-9 and abs(similarite([1, 2, 3], [2, 4, 6]) - 1) < 1e-9)
verifier("Exercice 2 · perpendiculaire = 0", lambda: abs(similarite([1, 0], [0, 1])) < 1e-9)
verifier("Exercice 2 · 45 degrés ≈ 0,707", lambda: abs(similarite([1, 1], [1, 0]) - 0.7071) < 1e-3)

<details><summary>Solution</summary>

```python
def similarite(a, b):
    produit = sum(x * y for x, y in zip(a, b))
    longueur_a = sum(x * x for x in a) ** 0.5
    longueur_b = sum(y * y for y in b) ** 0.5
    return produit / (longueur_a * longueur_b)

print(similarite([1, 0], [1, 0]), similarite([1, 0], [0, 1]), similarite([1, 2, 3], [2, 4, 6]), similarite([1, 1], [1, 0]))
# 1.0  0.0  1.0  0.707 : le cosinus regarde l'angle, pas la longueur des vecteurs.
```

</details>

## Exercice 3 ⭐ · Le mot le plus proche

Avec les embeddings 2D écrits à la main de la leçon, écris `plus_proche(mot)` qui renvoie le mot du dictionnaire (autre que lui-même) dont la similarité cosinus avec `mot` est la plus grande.

Résultat attendu : `plus_proche("hamster")` → `"souris"`, `plus_proche("voiture")` → `"camion"`.

<details><summary>Indice</summary>

`max((m for m in mots if m != mot), key=lambda m: similarite(mots[mot], mots[m]))`.

</details>

In [ ]:
# À toi
mots = {
    "chat": (0.9, 0.2), "chien": (0.9, 0.4), "lion": (0.95, 0.9), "souris": (0.85, 0.05),
    "pomme": (0.3, 0.1), "banane": (0.3, 0.15), "pastèque": (0.35, 0.5),
    "vélo": (0.05, 0.4), "voiture": (0.05, 0.7), "camion": (0.02, 0.95), "hamster": (0.9, 0.03),
}

def plus_proche(mot):
    return None

for m in ["hamster", "voiture", "chat"]:
    print(m, "→", plus_proche(m))

In [ ]:
verifier("Exercice 3 · plus_proche", lambda: plus_proche("hamster") == "souris" and plus_proche("voiture") == "camion" and plus_proche("vélo") == "voiture")

<details><summary>Solution</summary>

```python
mots = {
    "chat": (0.9, 0.2), "chien": (0.9, 0.4), "lion": (0.95, 0.9), "souris": (0.85, 0.05),
    "pomme": (0.3, 0.1), "banane": (0.3, 0.15), "pastèque": (0.35, 0.5),
    "vélo": (0.05, 0.4), "voiture": (0.05, 0.7), "camion": (0.02, 0.95), "hamster": (0.9, 0.03),
}

def plus_proche(mot):
    return max((m for m in mots if m != mot), key=lambda m: similarite(mots[mot], mots[m]))

for m in ["hamster", "voiture", "chat"]:
    print(m, "→", plus_proche(m))
# Surprise : chat → pomme ! Le cosinus ne regarde que l'angle : (0.9, 0.2) et (0.3, 0.1) pointent presque
# dans la même direction. En 2 dimensions c'est trompeur ; avec 384 dimensions, ça marche bien mieux.
```

</details>

## Exercice 4 ⭐ · Ctrl+F : chercher un mot

Avant les vecteurs, la recherche la plus simple : écris `chunks_contenant(mot)` qui renvoie la liste des **indices** des chunks qui contiennent le mot (majuscules ignorées).

Résultat attendu : `chunks_contenant("Mouette")` → `[1, 5, 8]`, `chunks_contenant("Japon")` → `[]`.

<details><summary>Indice</summary>

`[i for i, c in enumerate(chunks) if mot.lower() in c.lower()]`.

</details>

In [ ]:
# À toi
def chunks_contenant(mot):
    return None

print(chunks_contenant("Mouette"), chunks_contenant("tempête"), chunks_contenant("Japon"))

In [ ]:
verifier("Exercice 4 · chunks_contenant", lambda: chunks_contenant("Mouette") == [1, 5, 8] and chunks_contenant("tempête") == [1, 6] and chunks_contenant("Japon") == [])

<details><summary>Solution</summary>

```python
def chunks_contenant(mot):
    return [i for i, c in enumerate(chunks) if mot.lower() in c.lower()]

print(chunks_contenant("Mouette"), chunks_contenant("tempête"), chunks_contenant("Japon"))   # [1, 5, 8] [1, 6] []
# Limite : « piocher deux cartes » ne trouvera jamais « Mouette ». Il faut chercher par idée, pas par mot exact.
```

</details>

## Exercice 5 ⭐⭐ · Des chunks avec chevauchement

Quand un texte n'a pas de paragraphes, on découpe en **fenêtres de mots** qui se **chevauchent** pour ne pas couper une idée en deux. Écris `decouper_chevauchement(texte, taille=40, pas=30)` : une fenêtre de `taille` mots qui commence tous les `pas` mots ; on s'arrête dès qu'une fenêtre atteint la fin du texte.

Résultat attendu : sur un texte de 100 mots `m1 ... m100`, 3 chunks (débuts : `m1`, `m31`, `m61`), les 10 derniers mots du premier = les 10 premiers du deuxième, et le dernier chunk finit par `m100`.

<details><summary>Indice</summary>

`mots = texte.split()`, puis `for debut in range(0, len(mots), pas): morceaux.append(" ".join(mots[debut:debut + taille])); if debut + taille >= len(mots): break`.

</details>

In [ ]:
# À toi
def decouper_chevauchement(texte, taille=40, pas=30):
    mots = texte.split()
    morceaux = []
    return morceaux

texte_test = " ".join(f"m{i}" for i in range(1, 101))
test = decouper_chevauchement(texte_test)
print(len(test), "chunks :", [c.split()[0] + " ... " + c.split()[-1] for c in test])
print(len(decouper_chevauchement(REGLES_DU_JEU)), "chunks pour les règles")

In [ ]:
verifier("Exercice 5 · 3 fenêtres", lambda: len(test) == 3 and [c.split()[0] for c in test] == ["m1", "m31", "m61"] and test[-1].split()[-1] == "m100")
verifier("Exercice 5 · chevauchement de 10 mots", lambda: test[0].split()[-10:] == test[1].split()[:10] and all(len(c.split()) <= 40 for c in test))
verifier("Exercice 5 · sur les règles", lambda: 5 <= len(decouper_chevauchement(REGLES_DU_JEU)) <= 20)

<details><summary>Solution</summary>

```python
def decouper_chevauchement(texte, taille=40, pas=30):
    mots = texte.split()
    morceaux = []
    for debut in range(0, len(mots), pas):
        morceaux.append(" ".join(mots[debut:debut + taille]))
        if debut + taille >= len(mots):      # la fenêtre a atteint la fin : on s'arrête
            break
    return morceaux

texte_test = " ".join(f"m{i}" for i in range(1, 101))
test = decouper_chevauchement(texte_test)
print(len(test), "chunks :", [c.split()[0] + " ... " + c.split()[-1] for c in test])   # m1...m40, m31...m70, m61...m100
print(len(decouper_chevauchement(REGLES_DU_JEU)), "chunks pour les règles")
```

</details>

## Exercice 6 ⭐⭐ · TF-IDF et les k meilleurs passages

Deuxième et troisième étapes : **vectoriser** et **chercher**. Construis `tfidf` (un `TfidfVectorizer(stop_words=STOP_FR)` ajusté sur `chunks`) et `vecteurs` (les chunks transformés), puis écris `chercher(question, k=3)` qui renvoie la liste des `k` couples `(indice, score)` les mieux classés, du meilleur au moins bon, avec un score arrondi à 3 décimales.

Résultat attendu : `chercher("Que fait la carte Mouette ?")` place le chunk 5 en premier ; `chercher("Combien de manches dans une partie complète ?")` place le chunk 9 en premier.

<details><summary>Indice</summary>

`scores = cosine_similarity(tfidf.transform([question]), vecteurs)[0]`, puis `scores.argsort()[::-1][:k]` donne les indices des meilleurs.

</details>

In [ ]:
# À toi
tfidf = None
vecteurs = None

def chercher(question, k=3):
    return None

for indice, score in chercher("Que fait la carte Mouette ?") or []:
    print(f"[{score}] chunk {indice} : {chunks[indice][:70]}...")

In [ ]:
verifier("Exercice 6 · Mouette → chunk 5", lambda: chercher("Que fait la carte Mouette ?")[0][0] == 5 and len(chercher("Que fait la carte Mouette ?")) == 3)
verifier("Exercice 6 · scores décroissants", lambda: [s for _, s in chercher("Que fait la carte Mouette ?", 5)] == sorted([s for _, s in chercher("Que fait la carte Mouette ?", 5)], reverse=True))
verifier("Exercice 6 · manches → chunk 9", lambda: chercher("Combien de manches dans une partie complète ?", 1)[0][0] == 9)

<details><summary>Solution</summary>

```python
tfidf = TfidfVectorizer(stop_words=STOP_FR).fit(chunks)     # le vocabulaire est appris sur nos chunks
vecteurs = tfidf.transform(chunks)

def chercher(question, k=3):
    scores = cosine_similarity(tfidf.transform([question]), vecteurs)[0]
    meilleurs = scores.argsort()[::-1][:k]
    return [(int(i), round(float(scores[i]), 3)) for i in meilleurs]

for indice, score in chercher("Que fait la carte Mouette ?"):
    print(f"[{score}] chunk {indice} : {chunks[indice][:70]}...")
```

</details>

## Exercice 7 ⭐⭐ · Construire le prompt avec le contexte

Quatrième étape : **injecter**. Écris `construire_prompt(passages)` qui renvoie le prompt système : la consigne de répondre en français, en 2 phrases, **UNIQUEMENT** à partir du contexte, la phrase de refus `REFUS` si la réponse n'y est pas, puis `Contexte :` suivi des passages, un par ligne, chacun précédé de `- `.

Résultat attendu : le texte contient « UNIQUEMENT », `REFUS`, « Contexte : » et chaque passage.

<details><summary>Indice</summary>

`contexte = "\n".join(f"- {p}" for p in passages)`, puis une f-string : `f"... UNIQUEMENT à partir du contexte ci-dessous. Si la réponse n'y est pas, dis : {REFUS}\nContexte :\n{contexte}"`.

</details>

In [ ]:
# À toi
def construire_prompt(passages):
    return None

exemple = construire_prompt([chunks[5], chunks[8]]) if chunks else None
print(exemple)

In [ ]:
verifier("Exercice 7 · consignes", lambda: "UNIQUEMENT" in exemple and REFUS in exemple and "Contexte :" in exemple)
verifier("Exercice 7 · passages injectés", lambda: all(f"- {p}" in exemple for p in [chunks[5], chunks[8]]) and exemple.index("Contexte :") < exemple.index(chunks[5]))

<details><summary>Solution</summary>

```python
def construire_prompt(passages):
    contexte = "\n".join(f"- {p}" for p in passages)
    return ("Tu réponds en français, en 2 phrases maximum, UNIQUEMENT à partir du contexte ci-dessous. "
            f"Si la réponse n'y est pas, dis : {REFUS}\n"
            f"Contexte :\n{contexte}")

exemple = construire_prompt([chunks[5], chunks[8]])
print(exemple)
```

</details>

## Exercice 8 ⭐⭐ · Répondre avec le RAG

Cinquième étape : **répondre**. Écris `rag(question, k=3)` qui enchaîne tout : `chercher` → récupérer les textes des passages → `construire_prompt` → `demander(question, systeme=...)`.

Résultat attendu : « Combien de cartes reçoit chaque joueur au début ? » → une réponse qui contient « 7 cartes » ; « Qui a créé le jeu ? » → « Marchand ».

<details><summary>Indice</summary>

`passages = [chunks[i] for i, score in chercher(question, k)]`, puis `return demander(question, systeme=construire_prompt(passages))`.

</details>

In [ ]:
# À toi
def rag(question, k=3):
    return None

for q in ["Combien de cartes reçoit chaque joueur au début ?", "Qui a créé le jeu ?"]:
    print("Q :", q)
    print("   sans RAG :", demander(q))
    print("   avec RAG :", rag(q), "\n")

In [ ]:
verifier("Exercice 8 · 7 cartes", lambda: "7 cartes" in rag("Combien de cartes reçoit chaque joueur au début ?"))
verifier("Exercice 8 · Léa Marchand", lambda: "Marchand" in rag("Qui a créé le jeu ?"))

<details><summary>Solution</summary>

```python
def rag(question, k=3):
    passages = [chunks[i] for i, score in chercher(question, k)]
    return demander(question, systeme=construire_prompt(passages))

for q in ["Combien de cartes reçoit chaque joueur au début ?", "Qui a créé le jeu ?"]:
    print("Q :", q)
    print("   sans RAG :", demander(q))      # il invente
    print("   avec RAG :", rag(q), "\n")    # il lit la bonne page
```

</details>

## Exercice 9 ⭐⭐⭐ · Détecter « la réponse n'est pas dans mes documents »

Si la question n'a rien à voir avec les documents, le meilleur score est très bas : inutile d'appeler le modèle (et de risquer une hallucination). Écris :
1. `meilleur_score(question)` : le score du passage le mieux classé ;
2. `rag_avec_seuil(question, k=3, seuil=0.1)` : renvoie `REFUS` **sans appeler le modèle** si le meilleur score est sous le seuil, sinon la réponse de `rag`.

Résultat attendu : « Quelle est la capitale du Japon ? » → score 0 et `REFUS` ; « Combien de temps dure une partie ? » → une réponse avec « 15 minutes ».

<details><summary>Indice</summary>

`meilleur_score` : `chercher(question, 1)[0][1]`. Dans `rag_avec_seuil` : `if meilleur_score(question) < seuil: return REFUS`.

</details>

In [ ]:
# À toi
def meilleur_score(question):
    return None

def rag_avec_seuil(question, k=3, seuil=0.1):
    return None

for q in ["Quelle est la capitale du Japon ?", "Combien de temps dure une partie ?"]:
    print(f"[{meilleur_score(q)}] {q} → {rag_avec_seuil(q)}")

In [ ]:
verifier("Exercice 9 · meilleur_score", lambda: meilleur_score("Quelle est la capitale du Japon ?") == 0 and meilleur_score("Combien de temps dure une partie ?") > 0.3)
verifier("Exercice 9 · refus hors sujet", lambda: rag_avec_seuil("Quelle est la capitale du Japon ?") == REFUS)
verifier("Exercice 9 · réponse dans le sujet", lambda: "15 minutes" in rag_avec_seuil("Combien de temps dure une partie ?"))

<details><summary>Solution</summary>

```python
def meilleur_score(question):
    return chercher(question, 1)[0][1]

def rag_avec_seuil(question, k=3, seuil=0.1):
    if meilleur_score(question) < seuil:       # rien de pertinent : on n'appelle même pas le modèle
        return REFUS
    return rag(question, k)

for q in ["Quelle est la capitale du Japon ?", "Combien de temps dure une partie ?"]:
    print(f"[{meilleur_score(q)}] {q} → {rag_avec_seuil(q)}")
# Bonus : ça économise des tokens, donc du temps (ou de l'argent avec une API).
```

</details>

## Exercice 10 ⭐⭐⭐ · Évaluer le RAG sur 5 questions

Un RAG se mesure. Écris `evaluer(fonction, jeu_de_test)` : pour chaque couple `(question, attendu)`, elle appelle `fonction(question)` et compte un point si `attendu` apparaît dans la réponse (majuscules ignorées). Elle renvoie `(score, details)` où `details` est une liste de dictionnaires `{"question", "reponse", "ok"}`.

Résultat attendu : une fonction qui répond toujours « 7 cartes » marque 1 / 5 ; `rag_avec_seuil` marque 5 / 5 en mode démo (avec le vrai petit modèle, note ton score : c'est ta mesure de départ).

<details><summary>Indice</summary>

`ok = attendu.lower() in fonction(question).lower()`. Le score est la somme des `ok`.

</details>

In [ ]:
# À toi
jeu_de_test = [
    ("Combien de cartes reçoit chaque joueur au début ?", "7 cartes"),
    ("Qui a créé le jeu ?", "Marchand"),
    ("Combien de manches dans une partie complète ?", "3 manches"),
    ("Combien de temps dure une partie ?", "15 minutes"),
    ("Quelle est la capitale du Japon ?", "je ne trouve pas"),        # hors sujet : il doit refuser
]

def evaluer(fonction, jeu_de_test):
    return None

score, details = evaluer(rag_avec_seuil, jeu_de_test) or (None, [])
print("Score du RAG :", score, "/", len(jeu_de_test))
for d in details:
    print("  ", "✓" if d["ok"] else "✗", d["question"], "→", d["reponse"][:60])

In [ ]:
verifier("Exercice 10 · fonction bête = 1 / 5", lambda: evaluer(lambda q: "7 cartes", jeu_de_test)[0] == 1)
verifier("Exercice 10 · details", lambda: len(details) == 5 and set(details[0]) == {"question", "reponse", "ok"})
verifier("Exercice 10 · RAG = 5 / 5 (mode démo)", lambda: score == 5)

<details><summary>Solution</summary>

```python
jeu_de_test = [
    ("Combien de cartes reçoit chaque joueur au début ?", "7 cartes"),
    ("Qui a créé le jeu ?", "Marchand"),
    ("Combien de manches dans une partie complète ?", "3 manches"),
    ("Combien de temps dure une partie ?", "15 minutes"),
    ("Quelle est la capitale du Japon ?", "je ne trouve pas"),
]

def evaluer(fonction, jeu_de_test):
    details = []
    for question, attendu in jeu_de_test:
        reponse = fonction(question)
        details.append({"question": question, "reponse": reponse, "ok": attendu.lower() in reponse.lower()})
    return sum(d["ok"] for d in details), details

score, details = evaluer(rag_avec_seuil, jeu_de_test)
print("Score du RAG :", score, "/", len(jeu_de_test))
for d in details:
    print("  ", "✓" if d["ok"] else "✗", d["question"], "→", d["reponse"][:60])
```

</details>

## Exercice 11 ⭐⭐⭐ · Régler k

Le nombre `k` de passages injectés est un réglage : trop peu et la réponse manque, trop et le modèle se perd (et ça coûte des tokens). Calcule `scores_par_k`, un dictionnaire `k → score` pour `k` dans `[1, 2, 3, 5]` avec `evaluer` et `rag_avec_seuil`, puis `meilleur_k` (le plus **petit** `k` qui atteint le meilleur score).

Résultat attendu : un dictionnaire à 4 clés avec des scores entre 0 et 5, et `meilleur_k` cohérent avec lui.

<details><summary>Indice</summary>

`{k: evaluer(lambda q: rag_avec_seuil(q, k=k), jeu_de_test)[0] for k in [1, 2, 3, 5]}`. Pour le plus petit k au meilleur score : `min(k for k, s in scores_par_k.items() if s == max(scores_par_k.values()))`.

</details>

In [ ]:
# À toi
scores_par_k = None
meilleur_k = None
print(scores_par_k, "→ meilleur k :", meilleur_k)

In [ ]:
verifier("Exercice 11 · scores_par_k", lambda: sorted(scores_par_k) == [1, 2, 3, 5] and all(0 <= s <= 5 for s in scores_par_k.values()))
verifier("Exercice 11 · meilleur_k", lambda: meilleur_k == min(k for k, s in scores_par_k.items() if s == max(scores_par_k.values())))

<details><summary>Solution</summary>

```python
scores_par_k = {k: evaluer(lambda q: rag_avec_seuil(q, k=k), jeu_de_test)[0] for k in [1, 2, 3, 5]}
meilleur_score_possible = max(scores_par_k.values())
meilleur_k = min(k for k, s in scores_par_k.items() if s == meilleur_score_possible)
print(scores_par_k, "→ meilleur k :", meilleur_k)
# En mode démo, tous les k marchent : le faux modèle recopie la bonne phrase. Avec un vrai modèle,
# un k trop grand noie la bonne réponse dans du texte inutile.
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : un assistant sur tes notes

Assemble tout sur un **autre** document : `MES_NOTES` (le résumé des séances 9 à 11, que tu peux remplacer par ton propre texte). Écris `mon_assistant(question, k=2, seuil=0.1)` qui a **son propre index** (`notes_chunks`, `tfidf_notes`, `vecteurs_notes`), cherche les `k` meilleurs passages, refuse (`REFUS`) si le meilleur score est sous le seuil, sinon construit le prompt et répond.

Résultat attendu : « C'est quoi un token ? » → une réponse avec « morceau » ; « Quelles sont les étapes du RAG ? » → « découper » ; « Qui a gagné la Coupe du monde 2022 ? » → `REFUS`.

<details><summary>Indice</summary>

Refais les 3 lignes de l'index sur `notes_chunks` (un nouveau `TfidfVectorizer(stop_words=STOP_FR)`), puis recopie la logique de `rag_avec_seuil` en utilisant ce nouvel index.

</details>

In [ ]:
# À toi
MES_NOTES = """Séance 9 : un token est un morceau de mot transformé en nombre. Le modèle ne voit pas les lettres, c'est pour ça qu'il compte mal les r de strawberry.

Séance 9 : un LLM fait une seule chose, prédire le token suivant, encore et encore. La température règle le hasard : 0 = toujours pareil, élevée = créatif puis délirant.

Séance 9 : un LLM hallucine, c'est-à-dire qu'il invente une réponse plausible quand il ne sait pas. Il a une date de connaissance et il calcule mal.

Séance 10 : une API, c'est comme un serveur de restaurant. On envoie une commande (la requête) et on reçoit un plat (la réponse), souvent en JSON.

Séance 10 : les trois rôles sont system (les consignes), user (l'utilisateur) et assistant (le modèle). Le prompt système donne la personnalité et les règles.

Séance 10 : un chatbot n'a pas de mémoire, il renvoie tout l'historique des messages à chaque tour. Pour réutiliser une réponse dans un programme, on demande du JSON.

Séance 11 : le RAG donne au modèle les bons passages de mes documents avant de poser la question. Les étapes sont découper, vectoriser, chercher, injecter, répondre."""

notes_chunks = decouper(MES_NOTES)
tfidf_notes = None
vecteurs_notes = None

def mon_assistant(question, k=2, seuil=0.1):
    return None

for q in ["C'est quoi un token ?", "Quelles sont les étapes du RAG ?", "Qui a gagné la Coupe du monde 2022 ?"]:
    print("Q :", q)
    print("R :", mon_assistant(q), "\n")

In [ ]:
verifier("Exercice 12 · index des notes", lambda: vecteurs_notes.shape[0] == len(notes_chunks) == 7)
verifier("Exercice 12 · token → morceau", lambda: "morceau" in mon_assistant("C'est quoi un token ?").lower())
verifier("Exercice 12 · étapes du RAG", lambda: "découper" in mon_assistant("Quelles sont les étapes du RAG ?").lower())
verifier("Exercice 12 · hors sujet → refus", lambda: mon_assistant("Qui a gagné la Coupe du monde 2022 ?") == REFUS)

<details><summary>Solution</summary>

```python
MES_NOTES = """Séance 9 : un token est un morceau de mot transformé en nombre. Le modèle ne voit pas les lettres, c'est pour ça qu'il compte mal les r de strawberry.

Séance 9 : un LLM fait une seule chose, prédire le token suivant, encore et encore. La température règle le hasard : 0 = toujours pareil, élevée = créatif puis délirant.

Séance 9 : un LLM hallucine, c'est-à-dire qu'il invente une réponse plausible quand il ne sait pas. Il a une date de connaissance et il calcule mal.

Séance 10 : une API, c'est comme un serveur de restaurant. On envoie une commande (la requête) et on reçoit un plat (la réponse), souvent en JSON.

Séance 10 : les trois rôles sont system (les consignes), user (l'utilisateur) et assistant (le modèle). Le prompt système donne la personnalité et les règles.

Séance 10 : un chatbot n'a pas de mémoire, il renvoie tout l'historique des messages à chaque tour. Pour réutiliser une réponse dans un programme, on demande du JSON.

Séance 11 : le RAG donne au modèle les bons passages de mes documents avant de poser la question. Les étapes sont découper, vectoriser, chercher, injecter, répondre."""     # ou ton propre texte

notes_chunks = decouper(MES_NOTES)
tfidf_notes = TfidfVectorizer(stop_words=STOP_FR).fit(notes_chunks)
vecteurs_notes = tfidf_notes.transform(notes_chunks)

def mon_assistant(question, k=2, seuil=0.1):
    scores = cosine_similarity(tfidf_notes.transform([question]), vecteurs_notes)[0]
    meilleurs = scores.argsort()[::-1][:k]
    if scores[meilleurs[0]] < seuil:
        return REFUS
    passages = [notes_chunks[i] for i in meilleurs]
    return demander(question, systeme=construire_prompt(passages))

for q in ["C'est quoi un token ?", "Quelles sont les étapes du RAG ?", "Qui a gagné la Coupe du monde 2022 ?"]:
    print("Q :", q)
    print("R :", mon_assistant(q), "\n")
```

</details>

## Bravo !

Tu as construit un RAG complet et **mesurable** : découper (avec ou sans chevauchement), vectoriser (TF-IDF), chercher (top-k), injecter (le prompt), répondre, refuser quand il le faut, et évaluer sur un jeu de test. Remplace `MES_NOTES` par tes propres documents et refais tourner l'exercice 12.